# Leer el CSV de Aguas subterráneas

Este cuaderno carga y muestra las primeras filas del archivo `Aguas subterráneas.csv` ubicado en esta misma carpeta.

In [3]:
import pandas as pd

# Leer el CSV (ruta relativa a este cuaderno)
file_path = "Aguas subterráneas.csv"
# El fichero usa separador ';' y codificación Windows-1252 (latin-1), con coma decimal

df = pd.read_csv(
    file_path,
    sep=";",
    encoding="latin-1",
    decimal=",",
)

# Mostrar las primeras filas
print(f"Archivo cargado: {file_path}")
df.head()

Archivo cargado: Aguas subterráneas.csv


,Estación,Fecha Toma,Cod. Parámetro,Nom. Parámetro,Unidades,Valor numérico,Valor texto,Coord. X,Coord. Y,Cod Masa,Nombre Masa,Municipio,Provincia,UH Geo,UH Geo Nombre,Acuifero,Profundidad,Nombre C.A
0,CA0748-SIC01,08/10/2019,12DIBR,"1,2-DIBROMOETANO",µg/L,0,"< 1,0",697202,4215493,70042,TERCIARIO DE TORREVIEJA,Algorfa,ALICANTE,748,TERCIARIO DE TORREVIEJA,TERCIARIO DE TORREVIEJA,600,Comunidad Valenciana
1,CA0748-SIC01,08/10/2019,2BR2CLPRO,"1,2-DIBROMO-3-CLOROPROPANO",µg/L,0,"< 1,0",697202,4215493,70042,TERCIARIO DE TORREVIEJA,Algorfa,ALICANTE,748,TERCIARIO DE TORREVIEJA,TERCIARIO DE TORREVIEJA,600,Comunidad Valenciana
2,CA0748-SIC01,08/10/2019,2CLETA,"1,1-DICLOROETANO",µg/L,0,"< 1,0",697202,4215493,70042,TERCIARIO DE TORREVIEJA,Algorfa,ALICANTE,748,TERCIARIO DE TORREVIEJA,TERCIARIO DE TORREVIEJA,600,Comunidad Valenciana
3,CA0748-SIC01,08/10/2019,2CLETE,"1,1-DICLOROETENO",µg/L,0,"< 1,0",697202,4215493,70042,TERCIARIO DE TORREVIEJA,Algorfa,ALICANTE,748,TERCIARIO DE TORREVIEJA,TERCIARIO DE TORREVIEJA,600,Comunidad Valenciana
4,CA0748-SIC01,08/10/2019,2CLETE2,"1,2-DICLOROETENO",µg/L,0,"< 2,0",697202,4215493,70042,TERCIARIO DE TORREVIEJA,Algorfa,ALICANTE,748,TERCIARIO DE TORREVIEJA,TERCIARIO DE TORREVIEJA,600,Comunidad Valenciana


In [ ]:
# Leer el CSV (ruta relativa a este cuaderno)
file_path1 = "people_count.csv"
# El fichero usa separador ';' y codificación Windows-1252 (latin-1), con coma decimal

df1 = pd.read_csv(
    file_path1,
    sep=";",
    encoding="latin-1",
    decimal=",",
)

# Mostrar las primeras filas
print(f"Archivo cargado: {file_path1}")
df1.head()

Archivo cargado: people_count.csv


,ï»¿Fecha,count_in,count_out,Fecha_sin_hora,AÃ±o,Mes,Dia,Dia_semana,Hora,Minuto
0,2024-09-10 12:50:55,55,57,2024-09-10,2024,9,10,Martes,12,50
1,2024-09-10 12:51:34,55,57,2024-09-10,2024,9,10,Martes,12,51
2,2024-09-10 12:55:55,55,57,2024-09-10,2024,9,10,Martes,12,55
3,2024-09-10 12:56:50,55,57,2024-09-10,2024,9,10,Martes,12,56
4,2024-09-10 13:00:55,55,60,2024-09-10,2024,9,10,Martes,13,0


# Precio de la venta de viviendas en Torrevieja

In [21]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import time
import csv

# 1. Configuración del navegador indetectable
options = uc.ChromeOptions()
# Mantenemos la ventana visible para poder resolver el CAPTCHA si aparece
driver = uc.Chrome(options=options)

url = "https://www.idealista.com/sala-de-prensa/informes-precio-vivienda/venta/comunitat-valenciana/alicante-alacant/torrevieja/historico/"

try:
    print(f"Abriendo navegador en: {url}")
    driver.get(url)
    
    # --- MOMENTO CRÍTICO ---
    # Si te aparece un "Verificar que eres humano" o un puzzle, HAZLO TÚ MISMO en la ventana.
    # El script esperará 20 segundos para darte tiempo.
    print("Esperando 20 segundos para carga de página y posible resolución de CAPTCHA manual...")
    time.sleep(20) 

    # 2. Pasamos el HTML renderizado a BeautifulSoup
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    
    # Buscamos la tabla con las clases que identificamos en tu inspección
    table = soup.select_one('table.js-scroll-header.component-table')

    lista_precios = []

    if table:
        # Localizamos todas las filas de la tabla
        rows = table.find('tbody').find_all('tr', class_='table__row')
        
        for row in rows:
            # Buscamos las celdas de cada fila
            cells = row.find_all('td', class_='table__cell')
            
            if len(cells) >= 2:
                # Extracción según tu lógica original
                mes_raw = cells[0].get_text(strip=True)
                precio_raw = cells[1].get_text(strip=True)

                # Limpieza: "2.407 €/m2" -> "2407"
                # Quitamos la unidad, espacios y el punto de miles
                precio_limpio = precio_raw.replace("€/m2", "").replace(".", "").strip()
                
                lista_precios.append({
                    'Mes': mes_raw,
                    'Precio_m2': precio_limpio
                })

        # 3. Mostrar resultados en consola
        print(f"\n--- ÉXITO: Se han extraído {len(lista_precios)} meses ---")
        print(f"{'MES':<20} | {'PRECIO (€/m2)':<15}")
        print("-" * 40)
        for dato in lista_precios:
            print(f"{dato['Mes']:<20} | {dato['Precio_m2']:<15}")

        # 4. OPCIONAL: Guardar en un archivo CSV para Excel
        with open('precios_torrevieja.csv', 'w', newline='', encoding='utf-8') as file:
            writer = csv.DictWriter(file, fieldnames=['Mes', 'Precio_m2'])
            writer.writeheader()
            writer.writerows(lista_precios)
        print("\n[INFO] Los datos se han guardado en 'precios_torrevieja.csv'")

    else:
        print("ERROR: No se encontró la tabla. Probablemente el bloqueo persiste o la página no cargó.")

except Exception as e:
    print(f"Ocurrió un error inesperado: {e}")

finally:
    # Cerramos el navegador después de terminar
    print("Cerrando navegador en 5 segundos...")
    time.sleep(5)
    driver.quit()

Abriendo navegador en: https://www.idealista.com/sala-de-prensa/informes-precio-vivienda/venta/comunitat-valenciana/alicante-alacant/torrevieja/historico/
Esperando 20 segundos para carga de página y posible resolución de CAPTCHA manual...

--- ÉXITO: Se han extraído 242 meses ---
MES                  | PRECIO (€/m2)  
----------------------------------------
Febrero 2026         | 2407           
Enero 2026           | 2409           
Diciembre 2025       | 2392           
Noviembre 2025       | 2336           
Octubre 2025         | 2319           
Septiembre 2025      | 2282           
Agosto 2025          | 2262           
Julio 2025           | 2251           
Junio 2025           | 2236           
Mayo 2025            | 2185           
Abril 2025           | 2163           
Marzo 2025           | 2128           
Febrero 2025         | 2101           
Enero 2025           | 2079           
Diciembre 2024       | 2067           
Noviembre 2024       | 2042           
Octubre 2024  

# Precio de alquileres en Torrevieja

In [22]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import time
import csv

# 1. Configuración del navegador indetectable
options = uc.ChromeOptions()
driver = uc.Chrome(options=options)

# URL específica de ALQUILER (histórico)
url = "https://www.idealista.com/sala-de-prensa/informes-precio-vivienda/alquiler/comunitat-valenciana/alicante-alacant/torrevieja/historico/"

try:
    print(f"Accediendo a la sección de ALQUILER: {url}")
    driver.get(url)
    
    # RECUERDA: Si aparece el botón de "Pulsar y mantener", hazlo tú manualmente.
    print("Esperando 20 segundos para carga y bypass de seguridad...")
    time.sleep(20) 

    # 2. Procesamos el HTML con BeautifulSoup
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    table = soup.select_one('table.js-scroll-header.component-table')

    datos_alquiler = []

    if table:
        # Buscamos las filas de datos
        rows = table.find('tbody').find_all('tr', class_='table__row')
        
        for row in rows:
            cells = row.find_all('td', class_='table__cell')
            
            # Verificamos que la fila tenga datos reales (y no sea una fila vacía o de "n.d.")
            if len(cells) >= 2:
                mes = cells[0].get_text(strip=True)
                precio_raw = cells[1].get_text(strip=True) # Ejemplo: "12,3 €/m2"

                # Si el precio es "n.d.", lo saltamos o lo guardamos como vacío
                if "n.d." in precio_raw.lower():
                    continue

                # --- LIMPIEZA ESPECÍFICA PARA ALQUILER ---
                # 1. Quitamos " €/m2"
                # 2. Cambiamos la coma por punto para que sea un número (12,3 -> 12.3)
                precio_limpio = precio_raw.replace('€/m2', '').replace(',', '.').strip()
                
                datos_alquiler.append({
                    'Mes': mes,
                    'Precio_m2': precio_limpio
                })

        # 3. Guardado en CSV
        nombre_csv = 'alquiler_historico_torrevieja.csv'
        with open(nombre_csv, 'w', newline='', encoding='utf-16') as f:
            writer = csv.DictWriter(f, fieldnames=['Mes', 'Precio_m2'], delimiter='\t')
            writer.writeheader()
            writer.writerows(datos_alquiler)

        print(f"\n--- SCRAPING COMPLETADO ---")
        print(f"Se han guardado {len(datos_alquiler)} meses en '{nombre_csv}'")
        
        # Muestra los 5 primeros para verificar
        print("\nPrimeros registros extraídos:")
        for d in datos_alquiler[:5]:
            print(f"{d['Mes']}: {d['Precio_m2']} €/m2")

    else:
        print("No se pudo detectar la tabla. Revisa si el navegador muestra un error de acceso.")

except Exception as e:
    print(f"Error detectado: {e}")

finally:
    print("\nCerrando navegador...")
    time.sleep(4)
    driver.quit()

Accediendo a la sección de ALQUILER: https://www.idealista.com/sala-de-prensa/informes-precio-vivienda/alquiler/comunitat-valenciana/alicante-alacant/torrevieja/historico/
Esperando 20 segundos para carga y bypass de seguridad...

--- SCRAPING COMPLETADO ---
Se han guardado 213 meses en 'alquiler_historico_torrevieja.csv'

Primeros registros extraídos:
Febrero 2026: 12.3 €/m2
Enero 2026: 12.4 €/m2
Diciembre 2025: 12.1 €/m2
Noviembre 2025: 12.0 €/m2
Octubre 2025: 11.8 €/m2

Cerrando navegador...


# Leer archivos torrevieja

In [1]:
# pip install pandas odfpy
import pandas as pd

In [2]:
df_Crim_24_22 = pd.read_excel('Criminalidad_2024-2022.ods', engine='odf')
df_Crim_24_22.head(20)

,Delitos,2024-1T,2025-1T,2024-2T,2025-2T,2024-3T,2025-3T,2024-4T,2025-4T,2023-1T,2023-2T,2023-3T,2023-4T,2022-1T,2022-2T,2022-3T,2022-4T
0,I. CRIMINALIDAD CONVENCIONAL,1921,1893,3860,4344,6422,7122,6422,7122,1614,3587,6099,8041,1480,3283,5658,7371
1,1. Homicidios dolosos y asesinatos consumados,1,0,1,0,1,1,1,1,0,0,1,1,0,0,0,0
2,2. Homicidios dolosos y asesinatos en grad...,1,1,1,2,1,3,1,3,1,1,2,2,0,1,1,2
3,3. Delitos graves y menos graves de lesion...,36,32,84,67,145,108,145,108,27,55,100,146,17,43,86,104
4,4. Secuestro,0,0,0,0,1,0,1,0,0,0,1,2,0,0,1,1
5,5. Delitos contra la libertad sexual,16,22,39,44,68,67,68,67,16,33,56,73,15,33,61,74
6,5.1.-Agresión sexual con penetración,4,3,8,11,17,19,17,19,2,7,14,20,4,8,17,19
7,5.2.-Resto de delitos contra la libertad s...,12,19,31,33,51,48,51,48,14,26,42,53,11,25,44,55
8,6. Robos con violencia e intimidación,53,55,101,153,210,251,210,251,44,103,156,206,29,63,114,160
9,"7. Robos con fuerza en domicilios, estable...",136,142,272,313,441,509,441,509,121,237,376,499,123,294,485,611


In [3]:
df_Crim_20_21 = pd.read_excel('Criminalidad_2020-2021.ods', engine='odf')
df_Crim_20_21.head(20)

,Delitos,2021-1T,2021-2T,2021-3T,2021-4T,2020-1T,2020-2T,2020-3T,2020-4T
0,1.-Homicidios dolosos y asesinatos consumados,0,1,1,3,2,2,2,3
1,2.-Homicidios dolosos y asesinatos en grad...,4,5,8,8,0,0,2,3
2,3.-Delitos graves y menos graves de lesion...,9,27,71,98,21,39,69,84
3,4.-Secuestro,0,0,0,0,0,0,1,1
4,5.-Delitos contra la libertad e indemnidad...,6,25,40,56,14,23,36,47
5,5.1.-Agresión sexual con penetración,1,4,6,7,2,2,4,7
6,5.2.-Resto de delitos contra la libertad e...,5,21,34,49,12,21,32,40
7,6.-Robos con violencia e intimidación,20,37,76,103,44,51,91,125
8,"7.- Robos con fuerza en domicilios, establ...",150,312,490,621,216,324,553,679
9,7.1.-Robos con fuerza en domicilios,119,243,384,489,179,272,464,560


In [4]:
df_gastos_ingresos = pd.read_excel('Gastos-Ingresos_22-25.ods', engine='odf')
df_gastos_ingresos.head(20)

,Año y trimestre,Gastos corrientes en bienes y servicios,Gastos en inversiones,"Tasas, precios públicos y otros ingresos"
0,2022-1T,7059422.50,3210315.63,1426569.37
1,2022-2T,19181233.12,5982982.75,5846939.75
2,2022-3T,31854821.35,6591456.00,7084401.26
3,2022-4T,63309008.11,12029399.65,12992689.81
4,2023-1T,5482259.22,1102313.34,3187406.46
5,2023-2T,25327364.71,3238684.10,7106110.59
6,2023-3T,43756926.70,4392186.82,8604559.60
7,2023-4T,71796651.83,11217567.06,14383794.00
8,2024-1T,12000447.09,385455.19,2564571.24
9,2024-2T,22495424.51,8842158.37,8317313.76


In [5]:
df_tomas_playas = pd.read_excel('Tomas_playas.ods', engine='odf')
df_tomas_playas.head(20)

,Playa,Fecha toma,Escherichia coli,Enterococo,Observaciones
0,Playa Cala de las Piteras,2025-09-09,1 UFC/100 mL,1 NMP/100 mL,Zona Apta para el baño
1,Playa Cala de las Piteras,2025-09-02,1 UFC/100 mL,1 NMP/100 mL,Zona Apta para el baño
2,Playa Cala de las Piteras,2025-08-26,1 UFC/100 mL,1 NMP/100 mL,Zona Apta para el baño
3,Playa Cala de las Piteras,2025-08-19,2 UFC/100 mL,1 NMP/100 mL,Zona Apta para el baño
4,Playa Cala de las Piteras,2025-08-12,1 UFC/100 mL,10 NMP/100 mL,Zona Apta para el baño
5,Playa Cala de las Piteras,2025-08-05,1 UFC/100 mL,1 NMP/100 mL,Zona Apta para el baño
6,Playa Cala de las Piteras,2025-07-29,1 UFC/100 mL,10 NMP/100 mL,Zona Apta para el baño
7,Playa Cala de las Piteras,2025-07-22,1 UFC/100 mL,1 NMP/100 mL,Zona Apta para el baño
8,Playa Cala de las Piteras,2025-07-15,1 UFC/100 mL,10 NMP/100 mL,Zona Apta para el baño
9,Playa Cala de las Piteras,2025-07-08,2 UFC/100 mL,20 NMP/100 mL,Zona Apta para el baño


In [10]:
df_Aguas_subterraneas = pd.read_csv('Aguas subterráneas.csv', encoding="latin-1", sep=";")
df_Aguas_subterraneas.head(20)

,Estación,Fecha Toma,Cod. Parámetro,Nom. Parámetro,Unidades,Valor numérico,Valor texto,Coord. X,Coord. Y,Cod Masa,Nombre Masa,Municipio,Provincia,UH Geo,UH Geo Nombre,Acuifero,Profundidad,Nombre C.A
0,CA0748-SIC01,08/10/2019,12DIBR,"1,2-DIBROMOETANO",µg/L,0,"< 1,0",697202,4215493,70042,TERCIARIO DE TORREVIEJA,Algorfa,ALICANTE,748,TERCIARIO DE TORREVIEJA,TERCIARIO DE TORREVIEJA,600,Comunidad Valenciana
1,CA0748-SIC01,08/10/2019,2BR2CLPRO,"1,2-DIBROMO-3-CLOROPROPANO",µg/L,0,"< 1,0",697202,4215493,70042,TERCIARIO DE TORREVIEJA,Algorfa,ALICANTE,748,TERCIARIO DE TORREVIEJA,TERCIARIO DE TORREVIEJA,600,Comunidad Valenciana
2,CA0748-SIC01,08/10/2019,2CLETA,"1,1-DICLOROETANO",µg/L,0,"< 1,0",697202,4215493,70042,TERCIARIO DE TORREVIEJA,Algorfa,ALICANTE,748,TERCIARIO DE TORREVIEJA,TERCIARIO DE TORREVIEJA,600,Comunidad Valenciana
3,CA0748-SIC01,08/10/2019,2CLETE,"1,1-DICLOROETENO",µg/L,0,"< 1,0",697202,4215493,70042,TERCIARIO DE TORREVIEJA,Algorfa,ALICANTE,748,TERCIARIO DE TORREVIEJA,TERCIARIO DE TORREVIEJA,600,Comunidad Valenciana
4,CA0748-SIC01,08/10/2019,2CLETE2,"1,2-DICLOROETENO",µg/L,0,"< 2,0",697202,4215493,70042,TERCIARIO DE TORREVIEJA,Algorfa,ALICANTE,748,TERCIARIO DE TORREVIEJA,TERCIARIO DE TORREVIEJA,600,Comunidad Valenciana
5,CA0748-SIC01,08/10/2019,2CLETEN,"TRANS-1,2-DICLOROETENO",µg/L,0,"< 1,0",697202,4215493,70042,TERCIARIO DE TORREVIEJA,Algorfa,ALICANTE,748,TERCIARIO DE TORREVIEJA,TERCIARIO DE TORREVIEJA,600,Comunidad Valenciana
6,CA0748-SIC01,08/10/2019,2CLPRO,"1,3-DICLOROPROPANO",µg/L,0,"< 1,0",697202,4215493,70042,TERCIARIO DE TORREVIEJA,Algorfa,ALICANTE,748,TERCIARIO DE TORREVIEJA,TERCIARIO DE TORREVIEJA,600,Comunidad Valenciana
7,CA0748-SIC01,08/10/2019,2CLPROCIS,"CIS-1,3-DICLOROPROPENO",µg/L,0,"< 1,0",697202,4215493,70042,TERCIARIO DE TORREVIEJA,Algorfa,ALICANTE,748,TERCIARIO DE TORREVIEJA,TERCIARIO DE TORREVIEJA,600,Comunidad Valenciana
8,CA0748-SIC01,08/10/2019,2CLPROPA,"1,2-DICLOROPROPANO",µg/L,0,"< 1,0",697202,4215493,70042,TERCIARIO DE TORREVIEJA,Algorfa,ALICANTE,748,TERCIARIO DE TORREVIEJA,TERCIARIO DE TORREVIEJA,600,Comunidad Valenciana
9,CA0748-SIC01,08/10/2019,2CLPROPA2,"2,2-DICLOROPROPANO",µg/L,0,"< 1,0",697202,4215493,70042,TERCIARIO DE TORREVIEJA,Algorfa,ALICANTE,748,TERCIARIO DE TORREVIEJA,TERCIARIO DE TORREVIEJA,600,Comunidad Valenciana


In [16]:
import pandas as pd

# Definimos el nombre de tu archivo
file_name = 'alquiler_historico_torrevieja.csv'

# Para este archivo usamos:
# sep='\t' -> porque el separador es un tabulador
# encoding='utf-16' -> porque es el formato que usamos para proteger tildes y ñ
df = pd.read_csv(file_name, sep='\t', encoding='utf-16')

# Mostramos las primeras 5 filas
print("--- Datos de Alquiler en Torrevieja ---")
print(df.head())

# Bonus: Si quieres ver estadísticas rápidas (precio medio, máximo, etc.)
# print(df.describe())

--- Datos de Alquiler en Torrevieja ---
              Mes  Precio_m2
0    Febrero 2026       12.3
1      Enero 2026       12.4
2  Diciembre 2025       12.1
3  Noviembre 2025       12.0
4    Octubre 2025       11.8


In [17]:
import pandas as pd

# 1. Cargamos el archivo de venta
# Este archivo no necesita especificar encoding utf-16 ni separador especial
df_venta = pd.read_csv('precios_torrevieja.csv')

# 2. Mostramos las primeras filas
print("--- Histórico de Precios de Venta (Torrevieja) ---")
print(df_venta.head())

# 3. Tip: Como este archivo tiene algunos valores "nd" (no disponibles), 
# podemos ver cuántos hay así:
# print(df_venta[df_venta['Precio_m2'] == 'nd'])

--- Histórico de Precios de Venta (Torrevieja) ---
              Mes Precio_m2
0    Febrero 2026      2407
1      Enero 2026      2409
2  Diciembre 2025      2392
3  Noviembre 2025      2336
4    Octubre 2025      2319


### CALIDAD AGUAS DE BAÑO CON WEB SCRAPPING

In [26]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 1. Configuración de la URL y los datos del formulario (extraídos de tu HTML)
url = "https://nayadeciudadano.sanidad.gob.es/Splayas/ciudadano/ciudadanoVerZonaAction.do"

payload = {
    'pestanya': '3',                # Pestaña de Muestreos
    'actionProcedencia': 'ciudadanoListaZonaAction',
    'codZona': '1786',              # ID de Cala de las Piteras
    'codCCAA': '10',                # C. Valenciana
    'codProvincia': '3',            # Alicante
    'codMunicipio': '3133',         # Torrevieja
    'provinciaMapa': '',
    'denZona': '',
    'codPlaya': ''
}

# 2. Creamos una sesión para que la web no nos expulse (manejo de cookies)
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
})

try:
    # 3. Hacemos la petición POST simulando el clic en la pestaña "Muestreos"
    response = session.post(url, data=payload)
    response.raise_for_status() # Si hay error 403 o 500, saltará aquí

    # 4. Parseamos el resultado
    soup = BeautifulSoup(response.text, 'html.parser')
    celdas = soup.find_all('td', class_='valorCampoI')

    # 5. Agrupamos de 4 en 4 (Fecha, E.Coli, Enterococo, Observaciones)
    datos_limpios = []
    for i in range(0, len(celdas), 4):
        fila = [c.get_text(strip=True) for c in celdas[i:i+4]]
        if len(fila) == 4:
            datos_limpios.append(fila)

    # 6. Creamos el DataFrame y guardamos
    columnas = ["Fecha", "E_Coli", "Enterococo", "Estado"]
    df = pd.DataFrame(datos_limpios, columns=columnas)
    
    print(df)
    df.to_csv("muestreos_piteras.csv", index=False, encoding='utf-8-sig')
    print("\n✅ Proceso completado. Datos guardados en 'muestreos_piteras.csv'")

except Exception as e:
    print(f"❌ Error al conectar: {e}")

         Fecha         E_Coli     Enterococo                  Estado
0   09/09/2025   1 UFC/100 mL   1 NMP/100 mL  Zona Apta para el baño
1   02/09/2025   1 UFC/100 mL   1 NMP/100 mL  Zona Apta para el baño
2   26/08/2025   1 UFC/100 mL   1 NMP/100 mL  Zona Apta para el baño
3   19/08/2025   2 UFC/100 mL   1 NMP/100 mL  Zona Apta para el baño
4   12/08/2025   1 UFC/100 mL  10 NMP/100 mL  Zona Apta para el baño
5   05/08/2025   1 UFC/100 mL   1 NMP/100 mL  Zona Apta para el baño
6   29/07/2025   1 UFC/100 mL  10 NMP/100 mL  Zona Apta para el baño
7   22/07/2025   1 UFC/100 mL   1 NMP/100 mL  Zona Apta para el baño
8   15/07/2025   1 UFC/100 mL  10 NMP/100 mL  Zona Apta para el baño
9   08/07/2025   2 UFC/100 mL  20 NMP/100 mL  Zona Apta para el baño
10  01/07/2025   5 UFC/100 mL   1 NMP/100 mL  Zona Apta para el baño
11  20/06/2025   1 UFC/100 mL   1 NMP/100 mL  Zona Apta para el baño
12  10/06/2025   9 UFC/100 mL   9 NMP/100 mL  Zona Apta para el baño
13  27/05/2025   9 UFC/100 mL   9 

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 1. Lista de playas con sus códigos
playas = [
    {"nombre": "PLAYA DE CABO CERBERA", "codZona": "845"},
    {"nombre": "PLAYA DE CALA MAR AZUL", "codZona": "1785"},
    {"nombre": "PLAYA DE LA MATA", "codZona": "2279"},
    {"nombre": "PLAYA DE LAS PISCINAS DEL PASEO", "codZona": "846"},
    {"nombre": "PLAYA DE LOS LOCOS", "codZona": "842"},
    {"nombre": "PLAYA DE TORRELAMATA", "codZona": "841"},
    {"nombre": "PLAYA DEL CURA", "codZona": "843"},
    {"nombre": "PLAYA LOS NAUFRAGOS", "codZona": "844"},
    {"nombre": "CALA DE LAS PITERAS", "codZona": "1786"}  # ya estaba en tu código original
]

# 2. URL de la acción del formulario
url = "https://nayadeciudadano.sanidad.gob.es/Splayas/ciudadano/ciudadanoVerZonaAction.do"

# 3. Configuración de la sesión
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
})

# 4. Lista para guardar todos los datos
todos_datos = []

# 5. Iteramos por cada playa
for playa in playas:
    payload = {
        'pestanya': '3',  # Pestaña de Muestreos
        'actionProcedencia': 'ciudadanoListaZonaAction',
        'codZona': playa["codZona"],
        'codCCAA': '10',  # Comunidad Valenciana
        'codProvincia': '3',  # Alicante
        'codMunicipio': '3133',  # Torrevieja
        'provinciaMapa': '',
        'denZona': '',
        'codPlaya': ''
    }

    try:
        response = session.post(url, data=payload)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')
        celdas = soup.find_all('td', class_='valorCampoI')

        # Agrupamos de 4 en 4: Fecha, E.Coli, Enterococo, Observaciones
        datos_limpios = []
        for i in range(0, len(celdas), 4):
            fila = [c.get_text(strip=True) for c in celdas[i:i+4]]
            if len(fila) == 4:
                fila.insert(0, playa["nombre"])  # agregamos el nombre de la playa
                datos_limpios.append(fila)

        todos_datos.extend(datos_limpios)

        print(f"✅ Datos de {playa['nombre']} recogidos correctamente.")

    except Exception as e:
        print(f"❌ Error al conectar con {playa['nombre']}: {e}")

# 6. Guardamos todos los datos en un CSV
columnas = ["Playa", "Fecha", "E_Coli", "Enterococo", "Estado"]
df = pd.DataFrame(todos_datos, columns=columnas)
df.to_csv("muestreos_playas.csv", index=False, encoding='utf-8-sig')
print("\n✅ Proceso completado. Todos los datos guardados en 'muestreos_playas.csv'")

✅ Datos de PLAYA DE CABO CERBERA recogidos correctamente.
✅ Datos de PLAYA DE CALA MAR AZUL recogidos correctamente.
✅ Datos de PLAYA DE LA MATA recogidos correctamente.
✅ Datos de PLAYA DE LAS PISCINAS DEL PASEO recogidos correctamente.
✅ Datos de PLAYA DE LOS LOCOS recogidos correctamente.
✅ Datos de PLAYA DE TORRELAMATA recogidos correctamente.
✅ Datos de PLAYA DEL CURA recogidos correctamente.
✅ Datos de PLAYA LOS NAUFRAGOS recogidos correctamente.
✅ Datos de CALA DE LAS PITERAS recogidos correctamente.

✅ Proceso completado. Todos los datos guardados en 'muestreos_playas.csv'
